# DeepNorm

源码导航：[core/norm/deep_norm.py](../../../core/norm/deep_norm.py) 中的 `DeepNorm` 与 `get_deepnorm_constants`。

Wang et al. (2022) 在 *DeepNet: Scaling Transformers to 1,000 Layers* 中提出 **DeepNorm**，通过**深度相关的残差缩放**与**权重初始化约束**，首次实现了 1,000 层 Transformer 的稳定训练。DeepNorm 并非传统意义上的独立归一化函数，而是一套将 Post-LayerNorm、残差放大与权重缩小相结合的**联合稳定方案**，成功桥接了 Post-LN 的性能优势与 Pre-LN 的训练稳定性。

### 1. 理论推导

DeepNorm 对标准 Transformer 残差块进行两处修改：

**（1）残差分支放大**

将残差连接前的输入 $x_l$ 乘以一个深度相关的常数 $\alpha > 1$：

$$x_{l+1} = \text{LayerNorm}\bigl(\alpha \cdot x_l + G_l(x_l)\bigr)$$

其中 $G_l$ 为第 $l$ 层的子层（Attention 或 FFN）。放大残差分支使得 skip connection 在深层中始终占主导地位，抑制子层变换的累积漂移。

**（2）权重初始化缩小**

对 Xavier 初始化后的特定权重矩阵再乘以深度相关常数 $\beta < 1$：

- **缩小的矩阵**：FFN 的中间/输出投影、Attention 的 Value (V) 投影与 Output (O) 投影。
- **保持的矩阵**：Query (Q) 与 Key (K) 投影不变。

理论上，该方案保证所有子层更新的总和为 $O(1)$，与网络深度无关，从而避免训练初期输出分布的爆炸性漂移。

**深度常数 $\alpha$ 与 $\beta$：**

| 架构 | $\alpha$ | $\beta$ |
|---|---|---|
| Encoder-only ($N$ 层) | $(2N)^{1/4}$ | $(8N)^{-1/4}$ |
| Decoder-only ($M$ 层) | $(2M)^{1/4}$ | $(8M)^{-1/4}$ |
| Encoder-Decoder (Enc) | $0.81(N^4 M)^{1/16}$ | $0.87(N^4 M)^{-1/16}$ |
| Encoder-Decoder (Dec) | $(3M)^{1/4}$ | $(12M)^{-1/4}$ |

其中系数 2 与 3 分别来源于每层包含 2 个（Encoder）或 3 个（Decoder，含 cross-attention）残差子层。

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.norm.deep_norm import DeepNorm, get_deepnorm_constants

### 2. 深度常数计算

In [2]:
for n_layer in [12, 24, 48, 100, 1000]:
    alpha, beta = get_deepnorm_constants(n_layer, arch_type="decoder")
    print(f"n_layer={n_layer:4d}:  α={alpha:.4f},  β={beta:.4f}")

n_layer=  12:  α=2.2134,  β=0.3195
n_layer=  24:  α=2.6321,  β=0.2686
n_layer=  48:  α=3.1302,  β=0.2259
n_layer= 100:  α=3.7606,  β=0.1880
n_layer=1000:  α=6.6874,  β=0.1057


可见随着层数增加，$\alpha$ 缓慢增大（残差分支更强），而 $\beta$ 迅速减小（子层权重更保守），从而维持深层信号稳定。

### 3. 形状与 dtype 检查

In [3]:
alpha = get_deepnorm_constants(12, arch_type="decoder")[0]
norm = DeepNorm(64, alpha=alpha)

x = torch.randn(2, 4, 64)
sub = torch.randn(2, 4, 64)
y = norm(x, sub)

print("x.shape  =", tuple(x.shape))
print("sub.shape=", tuple(sub.shape))
print("y.shape  =", tuple(y.shape))
assert y.shape == x.shape, "DeepNorm 必须保持输入输出维度一致！"

x.shape  = (2, 4, 64)
sub.shape= (2, 4, 64)
y.shape  = (2, 4, 64)


### 4. $\alpha$ 对残差路径的影响

In [4]:
torch.manual_seed(42)
x = torch.randn(1, 1, 8)
sub = torch.randn(1, 1, 8)

norm_a1 = DeepNorm(8, alpha=1.0, eps=1e-6)
norm_a2 = DeepNorm(8, alpha=2.0, eps=1e-6)

y1 = norm_a1(x, sub)
y2 = norm_a2(x, sub)

print("α=1.0 输出:", y1.squeeze().tolist())
print("α=2.0 输出:", y2.squeeze().tolist())
assert not torch.allclose(y1, y2, atol=1e-5), "不同 α 应产生不同输出！"

α=1.0 输出: [0.5100139379501343, 0.07039324194192886, 0.47833603620529175, 0.7738194465637207, -0.3763751983642578, -2.413386344909668, 0.9700823426246643, -0.012883219867944717]
α=2.0 输出: [0.4167030155658722, 0.028206946328282356, 0.33314841985702515, 0.5026627779006958, -1.0291434526443481, -1.6194852590560913, 1.8765240907669067, -0.5086167454719543]


### 5. 源码精讲

以下为 `core/norm/deep_norm.py` 的完整实现：

```python
def get_deepnorm_constants(n_layer: int, arch_type: str = "decoder") -> tuple[float, float]:
    """计算 DeepNorm 的深度相关常数 (α, β)。"""
    if arch_type == "encoder":
        alpha = (2 * n_layer) ** 0.25
        beta  = (8 * n_layer) ** -0.25
    elif arch_type == "decoder":
        alpha = (2 * n_layer) ** 0.25
        beta  = (8 * n_layer) ** -0.25
    elif arch_type == "encoder_decoder":
        ...  # 根据 Encoder/Decoder 层数联合计算
    return alpha, beta

class DeepNorm(nn.Module):
    def __init__(self, normalized_shape: int, alpha: float, eps: float = 1e-5, bias: bool = True):
        super().__init__()
        self.alpha = alpha  # 残差分支放大系数
        # 内部封装标准 LayerNorm，保留其 scale + bias
        self.layer_norm = nn.LayerNorm(normalized_shape, eps=eps,
                                       elementwise_affine=True, bias=bias)

    def forward(self, x: torch.Tensor, sublayer_out: torch.Tensor) -> torch.Tensor:
        # 对应公式：LayerNorm(α·x + sublayer_out)
        return self.layer_norm(self.alpha * x + sublayer_out)
```

关键设计点：
- `alpha` 在 `__init__` 时固定，不参与梯度更新；其值由网络深度决定。
- 内部复用 `nn.LayerNorm` 的成熟实现，仅需修改残差相加前的缩放。
- 用户需在模型初始化阶段，对 `o_proj.weight` 和 `down_proj.weight` 等执行 `*= beta`，本模块不负责此操作。

---

## 延伸阅读与参考资料

### 核心论文
- **DeepNet: Scaling Transformers to 1,000 Layers**: Wang et al., 2022. [arXiv:2203.00555](https://arxiv.org/abs/2203.00555)

### 工程实践
- **GLM-130B**: 采用 DeepNorm 实现千亿参数模型的稳定训练
- **FoundationLayerNorm**: 后续对 DeepNorm 在 BERT/GPT 上的扩展